<a href="https://colab.research.google.com/github/jaqueuchoab/dados-divas/blob/main/agrupamento_curso_enade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

In [ ]:
base = pd.read_parquet('/content/enade_compilado_2022.parquet')

print(f"Base carregada: {len(base)} linhas, {len(base.columns)} colunas")

Base carregada: 594013 linhas, 17 colunas


In [ ]:
CHAVE = ['CO_CURSO', 'CO_MUNIC_CURSO']

def agregar_valor_unico(df, coluna):
    """Para variáveis fixas do curso — retorna o primeiro valor do grupo."""
    return df.groupby(CHAVE)[coluna].first().rename(coluna)

def agregar_contagem_categorica(df, coluna, prefixo=None):
    """Para variáveis categóricas — conta ocorrências de cada categoria."""
    pref = prefixo or f'QT_{coluna}_'
    return (
        df.groupby(CHAVE + [coluna])[coluna]
        .count()
        .unstack(fill_value=0)
        .add_prefix(pref)
    )

def agregar_contagem_categorias_selecionadas(df, coluna, categorias_colunas):
    chaves = df[CHAVE].drop_duplicates()
    resultado = chaves.set_index(CHAVE)

    for categoria, nome_coluna in categorias_colunas.items():
        contagem = (
            df[df[coluna] == categoria]
            .groupby(CHAVE)[coluna]
            .count()
            .rename(nome_coluna)
        )
        if nome_coluna in resultado.columns:
            resultado[nome_coluna] = resultado[nome_coluna].add(contagem, fill_value=0)
        else:
            resultado = resultado.join(contagem, how='left')

    return resultado.fillna(0).astype(int)

def agregar_contagem_por_nivel(df, coluna, nome_base):
    """
    Conta ocorrências de notas por nível de intervalo por curso e município.

    Gera 4 colunas:
    QUANT_{nome_base}_NIVEL_1 → 0 a 25
    QUANT_{nome_base}_NIVEL_2 → 26 a 50
    QUANT_{nome_base}_NIVEL_3 → 51 a 75
    QUANT_{nome_base}_NIVEL_4 → 76 a 100
    """
    niveis = {
        f'QUANT_{nome_base}_NIVEL_1': (0, 25),
        f'QUANT_{nome_base}_NIVEL_2': (26, 50),
        f'QUANT_{nome_base}_NIVEL_3': (51, 75),
        f'QUANT_{nome_base}_NIVEL_4': (76, 100),
    }

    chaves = df[CHAVE].drop_duplicates()
    resultado = chaves.set_index(CHAVE)

    for nome_coluna, (minimo, maximo) in niveis.items():
        contagem = (
            df[(df[coluna] >= minimo) & (df[coluna] <= maximo)]
            .groupby(CHAVE)[coluna]
            .count()
            .rename(nome_coluna)
        )
        resultado = resultado.join(contagem, how='left')

    return resultado.fillna(0).astype(int)

In [ ]:
VALOR_UNICO = [
    'CO_IES'
]

CONTAGEM_CATEGORICA = [
    'CO_RS_I1',
    'CO_RS_I2',
]

categorias_tp_pr_ger = {
    222: 'QUANT_TP_GER_ALU_AUSENTE',
    333: 'QUANT_TP_GER_ALU_PROVA_BRANCO',
    555: 'QUANT_TP_GER_ALU_PROVA_VALIDA',
    565: 'QUANT_TP_GER_ALU_DISPENSADO',
    585: 'QUANT_TP_GER_ALU_DISPENSADO',
}

categorias_tp_pr_ob_fg = {
    222: 'QUANT_TP_OB_FG_ALU_AUSENTE',
    333: 'QUANT_TP_OB_FG_ALU_PROVA_BRANCO',
    555: 'QUANT_TP_OB_FG_ALU_PROVA_VALIDA',
    565: 'QUANT_TP_OB_FG_ALU_DISPENSADO',
    585: 'QUANT_TP_OB_FG_ALU_DISPENSADO',
}

categorias_tp_pr_di_fg = {
    222: 'QUANT_TP_DI_FG_ALU_AUSENTE',
    333: 'QUANT_TP_DI_FG_ALU_PROVA_BRANCO',
    555: 'QUANT_TP_DI_FG_ALU_PROVA_VALIDA',
    565: 'QUANT_TP_DI_FG_ALU_DISPENSADO',
    585: 'QUANT_TP_DI_FG_ALU_DISPENSADO',
}

categorias_tp_pr_ob_ce = {
    222: 'QUANT_TP_OB_CE_ALU_AUSENTE',
    333: 'QUANT_TP_OB_CE_ALU_PROVA_BRANCO',
    555: 'QUANT_TP_OB_CE_ALU_PROVA_VALIDA',
    565: 'QUANT_TP_OB_CE_ALU_DISPENSADO',
    585: 'QUANT_TP_OB_CE_ALU_DISPENSADO',
}

categorias_tp_pr_di_ce = {
    222: 'QUANT_TP_DI_CE_ALU_AUSENTE',
    333: 'QUANT_TP_DI_CE_ALU_PROVA_BRANCO',
    555: 'QUANT_TP_DI_CE_ALU_PROVA_VALIDA',
    565: 'QUANT_TP_DI_CE_ALU_DISPENSADO',
    585: 'QUANT_TP_DI_CE_ALU_DISPENSADO',
}

def construir_base_agrupada(base):
    partes = []

    for c in VALOR_UNICO:
        if c in base.columns:
            partes.append(agregar_valor_unico(base, c))

    for c in CONTAGEM_CATEGORICA:
        if c in base.columns:
            partes.append(agregar_contagem_categorica(base, c))

    partes.append(agregar_contagem_categorias_selecionadas(base, 'TP_PR_GER', categorias_tp_pr_ger))
    partes.append(agregar_contagem_categorias_selecionadas(base, 'TP_PR_OB_FG', categorias_tp_pr_ob_fg))
    partes.append(agregar_contagem_categorias_selecionadas(base, 'TP_PR_DI_FG', categorias_tp_pr_di_fg))
    partes.append(agregar_contagem_categorias_selecionadas(base, 'TP_PR_OB_CE', categorias_tp_pr_ob_ce))
    partes.append(agregar_contagem_categorias_selecionadas(base, 'TP_PR_DI_CE', categorias_tp_pr_di_ce))

    # contagem de notas por nível
    partes.append(agregar_contagem_por_nivel(base, 'NT_GER', 'NT_GER'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_FG',  'NT_FG'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_OBJ_FG',  'NT_OBJ_FG'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_DIS_FG',  'NT_DIS_FG'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_CE',  'NT_CE'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_OBJ_CE',  'NT_OBJ_CE'))
    partes.append(agregar_contagem_por_nivel(base, 'NT_DIS_CE',  'NT_DI_CE'))

    resultado = pd.concat(partes, axis=1)
    resultado.index.names = CHAVE  # ← atualizado para chave composta
    resultado = resultado.reset_index()

    return resultado

In [ ]:
base_agrupada = construir_base_agrupada(base)

print(f"Base agrupada: {len(base_agrupada)} linhas, {len(base_agrupada.columns)} colunas")
print(base_agrupada.head())

Base agrupada: 9896 linhas, 65 colunas
   CO_CURSO  CO_MUNIC_CURSO  CO_IES  QT_CO_RS_I1_*  QT_CO_RS_I1_.  \
0         1         5103403       1            0.0            0.0   
1         2         5103403       1            0.0            1.0   
2         7         5103403       1            0.0            0.0   
3         8         5103403       1            0.0            1.0   
4        13         5103403       1            0.0            0.0   

   QT_CO_RS_I1_A  QT_CO_RS_I1_B  QT_CO_RS_I1_C  QT_CO_RS_I1_D  QT_CO_RS_I1_E  \
0            0.0           12.0           11.0           13.0            2.0   
1            0.0           11.0            6.0            9.0            1.0   
2            0.0            7.0           14.0           19.0            6.0   
3            0.0           22.0           25.0           15.0            5.0   
4            2.0           14.0           17.0           13.0            3.0   

   ...  QUANT_NT_CE_NIVEL_3  QUANT_NT_CE_NIVEL_4  QUANT_NT_OBJ_CE

In [ ]:
print(f"Cursos únicos:      {base_agrupada['CO_CURSO'].nunique()}")
print(f"IES únicas:         {base_agrupada['CO_IES'].nunique()}")
print(f"Municípios únicos:  {base_agrupada['CO_MUNIC_CURSO'].nunique()}")

Cursos únicos:      9896
IES únicas:         1749
Municípios únicos:  782


In [ ]:
os.makedirs('data/interim', exist_ok=True)

base_agrupada.to_parquet('data/interim/enade_agrupado.parquet', index=False)
base_agrupada.to_csv('data/interim/enade_agrupado.csv', sep=';', encoding='utf-8-sig', index=False)